ReAct Agent (Reasoning + Acting):
Definition:

The ReAct framework combines reasoning and acting in a single loop to handle tasks. The agent uses natural language reasoning (thinking through steps) and task actions (performing tasks like calculations or data retrieval).

The agent uses a combination of reasoning steps to guide actions in real-time, using feedback from those actions to further inform the next step in reasoning.

How it works:

Step 1: The agent receives a question or task.
Step 2: It reasons aloud (in natural language) about how to solve it.
Step 3: Based on its reasoning, it takes actions (e.g., searching a database, calculating something).
Step 4: The results of these actions are integrated into its reasoning and may trigger further actions.
Step 5: It repeats the process until it arrives at a solution.
Key points:

Combines thinking and doing (reasoning and actions).
Performs iterative steps, updating its process based on action results.
Typically handles complex decision-making scenarios.
Example:

Original task: “Calculate the total number of apples in a basket if there are 4 baskets and 7 apples in each.”
Reasoning: "I need to multiply 4 by 7 to get the total number of apples."
Action: Perform the multiplication.
Result: "There are 28 apples."
Reasoning: "I am done."

In [16]:

from langchain.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from langchain.agents import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

![image.png](attachment:image.png)

In [ ]:
import os  
from dotenv import load_dotenv

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT

os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY

os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [3]:
SERPER_API_KEY = "0089c4c6f4468442ed7a0696708d79a63f568fca"

In [4]:
google_search = GoogleSerperAPIWrapper(serper_api_key=SERPER_API_KEY)

tools = [
    Tool(
        name="Google Search",
        func=google_search.run,
        description="Allows the user to search for information using Google.",
    ),
]

In [6]:
template =  """Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}"""


In [7]:
prompt = PromptTemplate.from_template(template)

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="Gemma2-9b-It")

In [10]:
search_agent = create_react_agent(llm,tools,prompt)

In [12]:
agent_executor = AgentExecutor(
    agent=search_agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
)

In [14]:
agent_executor.invoke({"input": "Where is the hometown of the 2025 cricket world cup ?"})



> Entering new AgentExecutor chain...
Thought: I need to find information about the 2025 Cricket World Cup.
Action: Google Search
Action Input: 2025 Cricket World Cup hostIndia beat New Zealand in the final of the ICC Champions Trophy 2025 in Dubai. The Kiwis, who were missing leading bowler Matt Henry due to injury, posted 251-7 batting first, but Rohit Sharma lead the chase with a half-century. Dubai is hosting the final after India's refusal to travel to host nation Pakistan.Thought: The observation states that Dubai is hosting the final of the 2025 Cricket World Cup.
Action: Google Search
Action Input: 2025 Cricket World Cup host countryDubai is hosting the final after India's refusal to travel to host nation Pakistan. India went undefeated in the tournament and were the only side to beat New Zealand in their group-stage fixture. Both teams have won the tournament previously, India in 2002 and 2013 and New Zealand in 2000.Thought: The observation states that Pakistan is the host 

{'input': 'Where is the hometown of the 2025 cricket world cup ?',
 'output': 'Pakistan',
 'intermediate_steps': [(AgentAction(tool='Google Search', tool_input='2025 Cricket World Cup host', log='Thought: I need to find information about the 2025 Cricket World Cup.\nAction: Google Search\nAction Input: 2025 Cricket World Cup host'),
   "India beat New Zealand in the final of the ICC Champions Trophy 2025 in Dubai. The Kiwis, who were missing leading bowler Matt Henry due to injury, posted 251-7 batting first, but Rohit Sharma lead the chase with a half-century. Dubai is hosting the final after India's refusal to travel to host nation Pakistan."),
  (AgentAction(tool='Google Search', tool_input='2025 Cricket World Cup host country', log='Thought: The observation states that Dubai is hosting the final of the 2025 Cricket World Cup.\nAction: Google Search\nAction Input: 2025 Cricket World Cup host country'),
   "Dubai is hosting the final after India's refusal to travel to host nation Pak

# ReAct with Custom tool

In [17]:
# Custom tool for the Agent 
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
  return fake_employees.get(name,"Employee not found")

In [18]:
# Custom tool for the Agent 
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [20]:
from langchain import hub

In [21]:
# Saved React Prompt in langchain hub, we could manually type the prompt as well.
prompt = hub.pull("hwchase17/react")

In [22]:

print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [23]:
tools = [get_employee_salary, get_employee_id]

In [24]:

agent = create_react_agent(llm,tools,prompt)

In [25]:
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)

In [26]:

agent_executor.invoke({"input":"What is the Salary of Evan?"})



> Entering new AgentExecutor chain...
Thought: I need to find Evan's employee ID first, then use that to get his salary.
Action: get_employee_id
Action Input: EvanE005Thought: I can now use Evan's employee ID to get his salary.
Action: get_employee_salary
Action Input: E00545000Thought: I now know the final answer
Final Answer: 45000 


> Finished chain.


{'input': 'What is the Salary of Evan?', 'output': '45000'}

Self-Ask with Search Agent:
Definition:

This approach allows the AI to ask itself follow-up questions when it doesn't immediately know the answer. It performs a recursive questioning process to break down complex queries into smaller, more manageable ones.
Once the AI generates these follow-up questions, it performs an external search (e.g., through Google or an internal database) to gather the necessary information before formulating a response.
How it works:

Step 1: Receive a complex question.
Step 2: The agent identifies sub-questions or follow-up questions.
Step 3: It performs a search or fetches answers to these questions from external resources (like a web search).
Step 4: The answers are aggregated to provide a complete response.
Key points:

Emphasizes question decomposition.
Relies on external search for sub-questions.
Useful for answering open-ended or broad questions where the answer is not immediately available.
Example:

Original question: “How many moons does Jupiter have?”
Sub-question: “What is Jupiter?” (search)
Sub-question: “What are moons?” (search)
Finally, it retrieves the answer: "Jupiter has 79 moons."

In [34]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

In [35]:

from langchain import hub
from langchain.agents import AgentExecutor, create_self_ask_with_search_agent
from langchain_community.tools.tavily_search import TavilyAnswer

In [36]:
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
        verbose=True
    )
]

In [37]:

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/self-ask-with-search")

print(prompt.template)

Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washin

In [38]:

# Construct the Self Ask With Search Agent
agent = create_self_ask_with_search_agent(llm, tools, prompt)

In [39]:
# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)

In [40]:
agent_executor.invoke(
    {"input": "which asian country has largest population?"}
)



> Entering new AgentExecutor chain...
No.
So the final answer is: India (or China, depending on the definition of "Asia" used. Clarification may be needed if the user is using a definition that excludes part or all of either country.)

> Finished chain.


{'input': 'which asian country has largest population?',
 'output': 'India (or China, depending on the definition of "Asia" used. Clarification may be needed if the user is using a definition that excludes part or all of either country.)'}